# 📘 Module 1: Traditional Machine Learning
## Enterprise Use Case — Telecom Customer Churn Prediction

---

> **Series:** Fine-Tuning & ML Fundamentals — From Zero to Enterprise  
> **Module:** 1 of 6  
> **Level:** Absolute Beginner → Intermediate  
> **Time to complete:** ~2–3 hours (read + run + experiment)

---

## 👋 Who Is This For?

This notebook is for anyone who:
- Knows what ChatGPT/Claude does but doesn't know **how ML models actually learn**
- Wants to apply for ML/AI jobs but feels shaky on the fundamentals
- Has heard terms like "logistic regression", "random forest", "accuracy" and wants to finally understand them
- Wants to see how a **real enterprise team** uses ML to solve a business problem

**You do NOT need a math degree.** Every formula is explained in plain English.

---

## 🗺️ What You Will Learn in This Module

| Topic | What It Means |
|---|---|
| What is ML? | The actual intuition behind how machines learn |
| The ML Workflow | 7 steps every ML project follows |
| Data Exploration (EDA) | Understanding your data before touching code |
| Data Preprocessing | Cleaning & preparing data (90% of real work) |
| 3 Algorithms | Logistic Regression, Decision Tree, Random Forest |
| Model Evaluation | How to measure if your model is actually good |
| Feature Importance | Which inputs matter most to the model |
| Enterprise Context | How companies use this in production |

---

## 🏢 Our Business Problem: Customer Churn

**Company:** TeleNova — a fictional telecom company (think Airtel, Verizon, Jio)  
**Problem:** Customers are leaving. Every customer that leaves costs the company money.  
**Question:** Can we predict *which customers are likely to leave* before they actually do?

### Why Does This Matter?

- Acquiring a new customer costs **5–7x more** than retaining an existing one
- If we know a customer is about to leave, we can offer them a discount or better plan
- This is called **Churn Prediction** and it is used by every major telecom, SaaS, and subscription company

**Churn** = when a customer stops using your service / cancels their subscription

---

## ⚙️ Setup — Install Required Libraries

Run this cell first. These are the tools we need.

In [ ]:
# Install required packages (only need to run once)
# If you're on Google Colab or a fresh environment, uncomment and run this:
# !pip install pandas numpy scikit-learn matplotlib seaborn

# Check versions to make sure everything is installed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import __version__ as sklearn_version

print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')
print(f'scikit-learn: {sklearn_version}')
print('✅ All libraries loaded successfully!')

---

# 📖 LESSON 1: What IS Machine Learning?

## The Simple Analogy

Imagine you've hired 1,000 new employees. Some leave within 3 months, some stay for years. After a year, you look back and notice patterns:

- Employees who commute >2 hours tend to leave
- Employees who never got a promotion in 2 years tend to leave
- Employees with a manager who gives feedback tend to stay

You **learned these patterns from past data**. Now, when a new employee joins, you can **predict** if they might leave based on these patterns.

**That is exactly what Machine Learning does.** Instead of YOU spotting the patterns, you give the data to an algorithm and it spots the patterns automatically.

---

## The 3 Types of ML You Must Know

### 1. Supervised Learning ← **What this module covers**
- You give the model labeled examples: "this customer left" / "this customer stayed"
- The model learns the pattern
- You use it to predict unlabeled new customers
- **Examples:** Spam detection, fraud detection, churn prediction, disease diagnosis

### 2. Unsupervised Learning
- No labels. You say: "find me groups in this data"
- The model clusters similar things together on its own
- **Examples:** Customer segmentation, anomaly detection, topic modeling

### 3. Reinforcement Learning
- An agent takes actions and gets rewards or penalties
- It learns to maximize rewards over time
- **Examples:** Chess engines, robot control, ChatGPT's RLHF training

---

## The 7-Step ML Workflow (Used in Every Enterprise Project)

```
1. Define the Problem       → What are we trying to predict?
2. Collect Data             → Where does the data come from?
3. Explore Data (EDA)       → What patterns exist? Any issues?
4. Preprocess Data          → Clean, encode, scale the data
5. Train a Model            → Let the algorithm learn the patterns
6. Evaluate the Model       → Is it actually good? How do we know?
7. Deploy & Monitor         → Use it in production, watch for drift
```

We will follow all 7 steps in this notebook.

---

### 📝 Key Terms Glossary

| Term | Layman Meaning |
|---|---|
| **Features (X)** | The inputs — columns the model uses to learn |
| **Label / Target (y)** | What we want to predict — churn = yes or no |
| **Training data** | Past examples the model learns from |
| **Test data** | New examples used to check if the model works |
| **Algorithm** | The mathematical recipe the model uses to learn |
| **Parameters** | The numbers the model adjusts during learning |
| **Prediction** | The model's output for a new input |
| **Accuracy** | % of predictions that were correct |

---

# 📊 STEP 1: Create & Load the Dataset

## Why Are We Creating Data Instead of Loading a CSV?

We're generating **realistic synthetic data** so this notebook works for everyone without downloading files. The data mirrors real telecom datasets used in production.

## What Data Does TeleNova Have?

Every telecom company has this kind of customer data in their CRM (Customer Relationship Management) system:

| Column | What It Means | Type |
|---|---|---|
| `customer_id` | Unique ID for each customer | ID (not used for ML) |
| `tenure_months` | How long they've been a customer | Number |
| `monthly_charges` | How much they pay per month | Number |
| `total_charges` | Total amount paid ever | Number |
| `num_support_calls` | How many times they called support | Number |
| `num_service_outages` | Internet/call outages they experienced | Number |
| `age` | Customer age | Number |
| `contract_type` | Month-to-month / 1-year / 2-year contract | Category |
| `payment_method` | How they pay (auto/manual/bank) | Category |
| `has_streaming` | Do they subscribe to streaming add-on? | Yes/No |
| `has_security_addon` | Do they have a security add-on? | Yes/No |
| `churn` | **DID THEY LEAVE?** This is what we predict | 0=No, 1=Yes |

In [ ]:
import numpy as np
import pandas as pd

# Fix the random seed so we get the same data every time we run this
# Think of it like fixing the shuffle of a deck of cards so it's reproducible
np.random.seed(42)

n = 5000  # 5000 customers

# --- Generate raw fields ---
tenure = np.random.randint(1, 72, size=n)           # 1 to 72 months
monthly = np.random.uniform(20, 120, size=n)        # $20 to $120/month
age = np.random.randint(18, 75, size=n)             # 18 to 75 years
support_calls = np.random.poisson(lam=2, size=n)    # most customers call 0-4 times
outages = np.random.poisson(lam=1, size=n)          # most customers have 0-2 outages

contract = np.random.choice(
    ['Month-to-Month', 'One-Year', 'Two-Year'],
    size=n,
    p=[0.55, 0.25, 0.20]   # most customers are on month-to-month (easier to leave)
)
payment = np.random.choice(
    ['Electronic Check', 'Mailed Check', 'Bank Transfer', 'Credit Card'],
    size=n,
    p=[0.35, 0.25, 0.20, 0.20]
)
has_streaming = np.random.choice([0, 1], size=n, p=[0.45, 0.55])
has_security = np.random.choice([0, 1], size=n, p=[0.60, 0.40])

# --- Build churn probability based on real business logic ---
# Each factor nudges the probability of churning up or down
churn_prob = (
    0.10                                            # base rate: 10% always leave
    + 0.30 * (contract == 'Month-to-Month')         # no commitment → higher churn
    - 0.15 * (contract == 'Two-Year')               # locked in → lower churn
    + 0.005 * support_calls                         # more calls → more frustrated
    + 0.05 * outages                                # outages hurt retention
    - 0.003 * tenure                                # longer customers are more loyal
    + 0.002 * monthly                               # higher bills → more price-sensitive
    - 0.05 * has_security                           # addon users are more engaged
    + 0.10 * (payment == 'Electronic Check')        # manual payers notice bills more
)

# Clip to valid probability range [0.02, 0.95]
churn_prob = np.clip(churn_prob, 0.02, 0.95)

# Generate actual churn labels from the probabilities
churn = (np.random.rand(n) < churn_prob).astype(int)

# Total charges = tenure * monthly (with a little noise for realism)
total_charges = tenure * monthly * np.random.uniform(0.95, 1.05, size=n)

# --- Assemble the DataFrame ---
df = pd.DataFrame({
    'customer_id':        [f'CUST_{i:05d}' for i in range(n)],
    'tenure_months':      tenure,
    'monthly_charges':    monthly.round(2),
    'total_charges':      total_charges.round(2),
    'num_support_calls':  support_calls,
    'num_service_outages':outages,
    'age':                age,
    'contract_type':      contract,
    'payment_method':     payment,
    'has_streaming':      has_streaming,
    'has_security_addon': has_security,
    'churn':              churn
})

print(f'Dataset shape: {df.shape}')
print(f'Churned customers: {df.churn.sum()} ({df.churn.mean()*100:.1f}%)')
print(f'Retained customers: {(df.churn==0).sum()} ({(df.churn==0).mean()*100:.1f}%)')
df.head(10)

### 📝 Note — What Just Happened?

We created 5,000 rows of customer data. The `churn` column is our **target** — the thing we want to predict.

Notice the churn rate is around **30-35%**. This is realistic — most telecom companies have 20-35% annual churn. This is called a **class imbalance** (more non-churners than churners) and we'll deal with it later.

---

# 🔍 STEP 2: Exploratory Data Analysis (EDA)

## Why EDA Before Modeling?

Before you build any model, you need to **understand your data**. Skipping EDA is the #1 mistake beginners make. Real enterprise data scientists spend 40-60% of their time just on EDA.

**What we're looking for:**
1. Missing values (data gaps)
2. Distribution of each feature (is anything weird?)
3. Relationship between features and the target (what predicts churn?)
4. Correlations between features (are any inputs redundant?)

In [ ]:
# 2.1 — Basic summary statistics
# This tells you: count, mean, min, max, and quartiles for every numeric column
print('=== BASIC INFO ===')
df.info()
print('\n=== MISSING VALUES ===')
print(df.isnull().sum())
print('\n=== SUMMARY STATISTICS ===')
df.describe().round(2)

### 📝 Reading the Summary Statistics

| Statistic | What It Tells You |
|---|---|
| **count** | How many non-null values (check for missing data) |
| **mean** | Average value — the center of the data |
| **std** | Standard deviation — how spread out the values are |
| **min/max** | The extremes — useful to spot outliers |
| **25%/50%/75%** | Quartiles — 25% of data is below 25%, etc. |

**What to look for:**  
- If `count` differs between columns → missing values  
- If `min` or `max` looks unrealistic → outliers or data errors  
- If `mean` and `50%` (median) are very different → the data is skewed

In [ ]:
# 2.2 — Visualize the churn distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart: how many churned vs. not
churn_counts = df['churn'].value_counts()
axes[0].bar(
    ['Retained (0)', 'Churned (1)'],
    churn_counts.values,
    color=['steelblue', 'tomato'],
    edgecolor='black'
)
axes[0].set_title('Customer Churn Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 30, f'{v}\n({v/n*100:.1f}%)', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(
    churn_counts.values,
    labels=['Retained', 'Churned'],
    autopct='%1.1f%%',
    colors=['steelblue', 'tomato'],
    startangle=90
)
axes[1].set_title('Churn Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Note: More customers are retained than churned — this is a class imbalance problem.')

In [ ]:
# 2.3 — How do numeric features differ between churned and retained customers?
# This is the most important EDA plot for classification problems

numeric_features = ['tenure_months', 'monthly_charges', 'num_support_calls',
                    'num_service_outages', 'age']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, feat in enumerate(numeric_features):
    retained = df[df['churn'] == 0][feat]
    churned  = df[df['churn'] == 1][feat]
    
    axes[i].hist(retained, bins=30, alpha=0.6, label='Retained', color='steelblue', density=True)
    axes[i].hist(churned,  bins=30, alpha=0.6, label='Churned',  color='tomato',   density=True)
    axes[i].set_title(f'{feat}', fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Density')
    axes[i].legend()
    
    # Annotate with means
    axes[i].axvline(retained.mean(), color='steelblue', linestyle='--', linewidth=1.5,
                    label=f'Retained mean: {retained.mean():.1f}')
    axes[i].axvline(churned.mean(), color='tomato', linestyle='--', linewidth=1.5,
                    label=f'Churned mean: {churned.mean():.1f}')

axes[-1].set_visible(False)  # hide the unused 6th subplot
plt.suptitle('Feature Distributions: Retained vs Churned Customers', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the mean difference for interpretation
print('=== Mean Comparison: Churned vs. Retained ===' )
print(f'{"Feature":<25} {"Retained Mean":>15} {"Churned Mean":>15} {"Difference":>12}')
print('-' * 70)
for feat in numeric_features:
    r_mean = df[df['churn']==0][feat].mean()
    c_mean = df[df['churn']==1][feat].mean()
    diff = c_mean - r_mean
    arrow = '↑' if diff > 0 else '↓'
    print(f'{feat:<25} {r_mean:>15.2f} {c_mean:>15.2f} {arrow} {abs(diff):>9.2f}')

### 📝 EDA Insight — What Do These Charts Tell You?

Look at the mean comparison table. This is **business intelligence** in action:

- **tenure_months ↓**: Churned customers have *shorter* tenure. New customers leave more.
- **monthly_charges ↑**: Churned customers pay *more* per month. High bills = more likely to shop around.
- **num_support_calls ↑**: Churned customers called support *more*. Frustrated customers leave.
- **num_service_outages ↑**: Churned customers had *more outages*. Poor service = churn.

Even **before building any model**, we can already give the business team actionable advice:
> "Focus retention on new customers with high bills who have called support multiple times."

This is why EDA is so valuable.

In [ ]:
# 2.4 — Churn rate by categorical features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contract type vs churn rate
contract_churn = df.groupby('contract_type')['churn'].mean().sort_values(ascending=False)
bars = axes[0].bar(contract_churn.index, contract_churn.values * 100,
                   color=['tomato', 'orange', 'steelblue'], edgecolor='black')
axes[0].set_title('Churn Rate by Contract Type', fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_ylim(0, 70)
for bar, val in zip(bars, contract_churn.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val*100:.1f}%', ha='center', fontweight='bold')

# Payment method vs churn rate
pay_churn = df.groupby('payment_method')['churn'].mean().sort_values(ascending=False)
colors = ['tomato', 'orange', 'steelblue', 'seagreen']
bars2 = axes[1].bar(pay_churn.index, pay_churn.values * 100, color=colors, edgecolor='black')
axes[1].set_title('Churn Rate by Payment Method', fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 70)
axes[1].tick_params(axis='x', rotation=15)
for bar, val in zip(bars2, pay_churn.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val*100:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('categorical_churn.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key finding: Month-to-Month customers churn at a MUCH higher rate than those on longer contracts!')
print('Business action: Offer discounts to convert Month-to-Month customers to 1-year contracts.')

In [ ]:
# 2.5 — Correlation heatmap (numeric features only)
# Correlation tells us: when feature A goes up, does feature B also go up?
# Values range from -1 (opposite) to 0 (no relation) to +1 (same direction)

numeric_df = df[numeric_features + ['churn']]
corr_matrix = numeric_df.corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # hide upper triangle
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5
)
plt.title('Correlation Matrix — Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Show the correlations with churn specifically
print('=== Correlation with CHURN (most important column) ===')
churn_corr = corr_matrix['churn'].drop('churn').sort_values(key=abs, ascending=False)
for feat, corr in churn_corr.items():
    direction = 'positively' if corr > 0 else 'negatively'
    print(f'{feat:<25} {corr:>7.3f}  — {direction} correlated with churn')

### 📝 Understanding Correlation

**Correlation = how much two things move together.**

- `+1.0` → perfect positive: when A goes up, B always goes up
- `-1.0` → perfect negative: when A goes up, B always goes down
- `0.0` → no relationship at all

**Why does it matter for ML?**
- Features highly correlated with `churn` are good predictors
- Features highly correlated with *each other* are redundant — the model doesn't need both

**Important:** Correlation only measures *linear* relationships. ML models can find non-linear patterns too.

---

# 🛠️ STEP 3: Data Preprocessing

## What Is Preprocessing and Why Is It Needed?

ML algorithms are mathematical. They work with **numbers**. But our data has:
- Text values: `'Month-to-Month'`, `'Electronic Check'`
- Numbers on very different scales: `age` (18–75) vs `total_charges` (20–8640)

We need to:
1. **Encode** text → numbers
2. **Scale** numbers → same range
3. **Split** data → training set and test set

### 🔑 The Most Important Rule in ML

> **Never let the model see the test data during training.**

This is like giving a student the exam answers while they study. Their "performance" in training would be great, but they wouldn't have actually learned anything. We test on data the model has **never seen** to measure real performance.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 3.1 — Drop the ID column (it has no predictive value)
df_ml = df.drop(columns=['customer_id'])

# 3.2 — Encode categorical features
# Technique: One-Hot Encoding
# 'Month-to-Month' becomes: month_to_month=1, one_year=0, two_year=0
# 'One-Year' becomes:       month_to_month=0, one_year=1, two_year=0
# This way the model doesn't think '2' > '1' (no false ordering)

df_encoded = pd.get_dummies(df_ml, columns=['contract_type', 'payment_method'], drop_first=False)

print('Columns after encoding:')
print(list(df_encoded.columns))
print(f'\nShape before encoding: {df_ml.shape}')
print(f'Shape after encoding:  {df_encoded.shape}')

### 📝 Two Ways to Encode Categories

**One-Hot Encoding (what we used):**
- Converts each category into its own binary column (0 or 1)
- Best when categories have **no natural order** (Month-to-Month is not "better" or "worse" than Two-Year, just different)
- Creates more columns, but prevents the model from assuming a numeric ordering

**Label Encoding:**
- Converts categories to integers: Month-to-Month=0, One-Year=1, Two-Year=2
- Best for **ordinal** categories where order matters (e.g., Low=0, Medium=1, High=2)
- ⚠️ Using label encoding for non-ordinal categories misleads the model

In [ ]:
# 3.3 — Separate features (X) from target (y)
X = df_encoded.drop(columns=['churn'])
y = df_encoded['churn']

print(f'Features (X): {X.shape}  — inputs the model uses to learn')
print(f'Target  (y): {y.shape}  — what the model tries to predict')
print(f'\nFeature names: {list(X.columns)}')

In [ ]:
# 3.4 — Split into training and test sets
# 80% of data → training (model learns from this)
# 20% of data → testing  (we evaluate the model on this, model never sees it during training)
# stratify=y ensures the churn ratio is the same in both splits (important for imbalanced data)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y           # keeps same churn % in train and test
)

print(f'Training set:  {X_train.shape[0]} rows ({X_train.shape[0]/n*100:.0f}%)')
print(f'Test set:      {X_test.shape[0]} rows ({X_test.shape[0]/n*100:.0f}%)')
print(f'\nChurn rate in training: {y_train.mean()*100:.1f}%')
print(f'Churn rate in test:     {y_test.mean()*100:.1f}%')
print('✅ Stratification worked — same churn rate in both splits!')

In [ ]:
# 3.5 — Feature Scaling
# Problem: 'age' ranges from 18-75, but 'total_charges' ranges from 20-8640
# Some algorithms treat larger numbers as more important — this is wrong!
# Solution: StandardScaler transforms each feature to mean=0, std=1
# After scaling: age and total_charges are on the same scale

scaler = StandardScaler()

# CRITICAL: Fit the scaler ONLY on training data
# Then apply (transform) to both train and test
# Why? Because in production, you won't have test data when you fit the scaler
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # just transform, NOT fit_transform

# Show before/after for 3 features
print('=== Before Scaling ===')
print(X_train[['tenure_months', 'monthly_charges', 'total_charges']].describe().round(2))

X_train_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
print('\n=== After Scaling (mean≈0, std≈1) ===')
print(X_train_df[['tenure_months', 'monthly_charges', 'total_charges']].describe().round(2))

### 📝 Which Algorithms Need Scaling?

| Algorithm | Needs Scaling? | Why? |
|---|---|---|
| Logistic Regression | ✅ Yes | Distance-based, larger numbers dominate |
| SVM | ✅ Yes | Distance-based |
| Neural Networks | ✅ Yes | Gradient descent sensitive to scale |
| Decision Trees | ❌ No | Splits based on thresholds, not distances |
| Random Forests | ❌ No | Ensemble of trees |
| XGBoost | ❌ No | Tree-based |

We'll scale for Logistic Regression and use unscaled for tree-based models.

---

# 🧠 STEP 4: Algorithm 1 — Logistic Regression

## What Is Logistic Regression?

Despite the name, **Logistic Regression is a classification algorithm**, not a regression one. It predicts the **probability** that a customer will churn.

### The Intuition

Imagine a weighing scale. Each feature gets a **weight** (coefficient):
- Positive weight → this feature increases churn probability
- Negative weight → this feature decreases churn probability

The model adds up: `(tenure × weight) + (monthly_charges × weight) + ...`

Then it squashes this sum through a **sigmoid function** that converts any number into a probability between 0 and 1.

```
If probability > 0.5 → Predict CHURN
If probability ≤ 0.5 → Predict RETAIN
```

### Why Start With Logistic Regression?

- Simple and fast
- Easy to interpret (you can read the weights)
- Good baseline — if a complex model barely beats it, is the complexity worth it?
- Used heavily in finance, healthcare (regulated industries that need explainability)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)

# Train the model
# max_iter=1000: give the optimizer enough steps to converge
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)    # <-- this is where learning happens

# Make predictions on the test set
lr_preds       = lr_model.predict(X_test_scaled)        # 0 or 1
lr_probs       = lr_model.predict_proba(X_test_scaled)[:, 1]  # probability of churn

# Basic accuracy
lr_accuracy = accuracy_score(y_test, lr_preds)
lr_auc      = roc_auc_score(y_test, lr_probs)

print(f'Logistic Regression Results:')
print(f'  Accuracy:  {lr_accuracy*100:.2f}%')
print(f'  ROC-AUC:   {lr_auc:.4f}')
print()
print('Detailed Report:')
print(classification_report(y_test, lr_preds, target_names=['Retained', 'Churned']))

### 📝 Understanding the Classification Report

This is the most important output. Let's decode each metric:

**The 4 types of predictions:**

```
                 Predicted: Retained    Predicted: Churned
Actual: Retained   True Negative (TN)    False Positive (FP)  ← Type I Error
Actual: Churned    False Negative (FN)   True Positive (TP)   ← Type II Error
                   ↑ missed churners!
```

**Precision** = Of all customers we predicted would churn, how many actually did?  
> High precision = fewer false alarms (you don't waste retention budget on loyal customers)

**Recall** = Of all customers who actually churned, how many did we catch?  
> High recall = fewer missed churners (you catch more customers before they leave)

**F1 Score** = Balance between precision and recall. Use this when both matter.

**For this business problem:**  
We care more about **recall** for churned customers. Missing a churner (False Negative) costs the company a customer. Falsely flagging a loyal customer for retention intervention is just a wasted discount — much cheaper.

In [ ]:
# 4.2 — Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, lr_preds)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=ax,
    xticklabels=['Predicted Retained', 'Predicted Churned'],
    yticklabels=['Actual Retained', 'Actual Churned']
)
ax.set_title('Confusion Matrix — Logistic Regression', fontweight='bold')

# Add explanation annotations
tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (TN): {tn} — correctly predicted retained')
print(f'False Positives (FP): {fp} — flagged as churn, but they stayed  (wasted discount)')
print(f'False Negatives (FN): {fn} — predicted retained, but they left  (missed churn!)')
print(f'True Positives  (TP): {tp} — correctly predicted churned')

print(f'\n💰 Business Impact:')
print(f'We correctly identified {tp}/{tp+fn} = {tp/(tp+fn)*100:.1f}% of actual churners')
print(f'We missed {fn} customers who left — each one is a lost revenue opportunity')

plt.tight_layout()
plt.savefig('confusion_matrix_lr.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4.3 — ROC Curve
# ROC = Receiver Operating Characteristic
# It shows: as you lower the decision threshold (from 0.5), you catch more churners
# but also get more false alarms
# AUC = Area Under the Curve: 0.5 = random guessing, 1.0 = perfect

fpr, tpr, thresholds = roc_curve(y_test, lr_probs)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'Logistic Regression (AUC = {lr_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Guess (AUC = 0.5)')
plt.xlabel('False Positive Rate (false alarms)')
plt.ylabel('True Positive Rate (churners caught)')
plt.title('ROC Curve — Logistic Regression', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.savefig('roc_curve_lr.png', dpi=150, bbox_inches='tight')
plt.show()

print('📖 How to read the ROC curve:')
print('  - Top-left corner = perfect model (catches all churners, zero false alarms)')
print('  - Diagonal line   = random guessing (no better than flipping a coin)')
print('  - The more area under the curve (AUC), the better the model')
print(f'  - Our AUC of {lr_auc:.3f} means we are significantly better than random')

---

# 🌳 STEP 5: Algorithm 2 — Decision Tree

## What Is a Decision Tree?

A Decision Tree works exactly like the guessing game **20 Questions**. The algorithm asks a series of yes/no questions to narrow down to a prediction.

**Example:**
```
Is tenure < 12 months?
├── YES → Is monthly charge > $80?
│         ├── YES → PREDICT: CHURN (high bill, new customer)
│         └── NO  → Is contract Month-to-Month?
│                   ├── YES → PREDICT: CHURN
│                   └── NO  → PREDICT: RETAIN
└── NO  → Is num_support_calls > 4?
          ├── YES → PREDICT: CHURN
          └── NO  → PREDICT: RETAIN
```

### How Does It Choose What to Ask?

The algorithm finds the question (split) that separates churners from non-churners **most cleanly**. It uses a metric called **Gini Impurity** or **Information Gain** to measure this.

### Advantages of Decision Trees
- Very easy to explain to business stakeholders (you can literally draw it)
- No scaling needed
- Handles non-linear patterns
- Fast to train

### Disadvantage
- **Overfitting**: A tree that is too deep memorizes the training data and fails on new data
- Solution: Limit depth with `max_depth`

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Train on UNSCALED data — trees don't need scaling
dt_model = DecisionTreeClassifier(
    max_depth=5,        # limit depth to prevent overfitting
    min_samples_leaf=20,  # each leaf must have at least 20 samples
    random_state=42
)
dt_model.fit(X_train, y_train)

dt_preds = dt_model.predict(X_test)
dt_probs = dt_model.predict_proba(X_test)[:, 1]
dt_accuracy = accuracy_score(y_test, dt_preds)
dt_auc      = roc_auc_score(y_test, dt_probs)

print(f'Decision Tree Results:')
print(f'  Accuracy: {dt_accuracy*100:.2f}%')
print(f'  ROC-AUC:  {dt_auc:.4f}')
print()
print(classification_report(y_test, dt_preds, target_names=['Retained', 'Churned']))

In [ ]:
# Visualize the top 3 levels of the tree
# This is what you'd show to a business stakeholder!
plt.figure(figsize=(20, 8))
plot_tree(
    dt_model,
    feature_names=X_train.columns.tolist(),
    class_names=['Retained', 'Churned'],
    filled=True,
    rounded=True,
    max_depth=3,          # show top 3 levels for readability
    fontsize=9
)
plt.title('Decision Tree — Top 3 Levels (Full tree has 5 levels)', fontweight='bold', fontsize=14)
plt.savefig('decision_tree.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Each node shows: the split condition, gini impurity, sample count, and class prediction')
print('   Blue nodes lean toward Retained, Orange nodes lean toward Churned')

In [ ]:
# Show what happens when we allow a deeper tree — overfitting example
print('=== Overfitting Demonstration ===')
print(f'{"Max Depth":<12} {"Train Accuracy":<18} {"Test Accuracy":<18} {"Overfitting?"}')
print('-' * 60)

for depth in [2, 5, 10, 20, None]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)
    train_acc = m.score(X_train, y_train)
    test_acc  = m.score(X_test, y_test)
    gap = train_acc - test_acc
    flag = '⚠️ YES' if gap > 0.05 else '✅ OK'
    depth_str = str(depth) if depth is not None else 'No limit'
    print(f'{depth_str:<12} {train_acc*100:>10.2f}%       {test_acc*100:>10.2f}%       {flag} (gap={gap:.3f})')

### 📝 Understanding Overfitting — The Core ML Problem

**Overfitting** is when a model learns the training data *too well* — including noise and random patterns that don't generalize.

Think of a student who memorizes every practice exam question but can't answer any new questions on the actual exam.

```
Underfitting:  model is too simple → misses patterns → bad on both train AND test
Good fit:      model captures real patterns → good on both train AND test
Overfitting:   model memorizes noise → great on train, bad on test
```

The table above shows: as depth increases past 5, training accuracy goes toward 100% but test accuracy stops improving — clear overfitting.

---

# 🌲 STEP 6: Algorithm 3 — Random Forest

## What Is a Random Forest?

A Random Forest is an **ensemble** (a committee) of Decision Trees. Instead of one tree making the decision, you train **hundreds of trees**, each on a slightly different random sample of the data. Then you take a vote.

**The Power of Committees:**  
One expert can be wrong. 500 experts, each trained slightly differently, voting together — that's much harder to fool.

### Why Is It Better Than a Single Tree?

- Each tree sees random subsets of rows AND random subsets of features
- No single tree can overfit the whole dataset
- Individual errors cancel out across trees
- This is called **Bagging** (Bootstrap Aggregating)

Random Forests are one of the **most reliable, widely-used algorithms in enterprise ML** because they:
- Rarely need tuning to perform well
- Handle missing values better
- Give you feature importance for free
- Work on tabular data (the most common type in enterprise)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,    # 300 trees in the forest
    max_depth=10,        # each tree can go up to depth 10
    min_samples_leaf=10, # each leaf needs at least 10 samples
    n_jobs=-1,           # use all CPU cores (faster training)
    random_state=42
)

print('Training Random Forest (300 trees)...')
rf_model.fit(X_train, y_train)
print('Done!')

rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_accuracy = accuracy_score(y_test, rf_preds)
rf_auc      = roc_auc_score(y_test, rf_probs)

print(f'\nRandom Forest Results:')
print(f'  Accuracy: {rf_accuracy*100:.2f}%')
print(f'  ROC-AUC:  {rf_auc:.4f}')
print()
print(classification_report(y_test, rf_preds, target_names=['Retained', 'Churned']))

---

# 📊 STEP 7: Compare All Models

Now we compare all three algorithms side by side — this is what you'd show a manager or in a technical presentation.

In [ ]:
# Side-by-side comparison
results = {
    'Logistic Regression': {'accuracy': lr_accuracy, 'auc': lr_auc, 'preds': lr_preds, 'probs': lr_probs},
    'Decision Tree':       {'accuracy': dt_accuracy, 'auc': dt_auc, 'preds': dt_preds, 'probs': dt_probs},
    'Random Forest':       {'accuracy': rf_accuracy, 'auc': rf_auc, 'preds': rf_preds, 'probs': rf_probs},
}

from sklearn.metrics import f1_score, recall_score, precision_score

print('='*75)
print(f'{"Model":<22} {"Accuracy":>10} {"AUC":>8} {"Precision":>11} {"Recall":>9} {"F1":>7}')
print('-'*75)
for name, res in results.items():
    p  = precision_score(y_test, res['preds'])
    r  = recall_score(y_test, res['preds'])
    f1 = f1_score(y_test, res['preds'])
    print(f'{name:<22} {res["accuracy"]*100:>9.2f}% {res["auc"]:>8.4f} {p*100:>10.2f}% {r*100:>8.2f}% {f1:>7.4f}')
print('='*75)
print('Note: Precision/Recall/F1 are for the Churned class (class=1)')

In [ ]:
# Visual comparison — ROC curves of all three models together
plt.figure(figsize=(8, 6))

colors = ['steelblue', 'darkorange', 'seagreen']
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['probs'])
    plt.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={res["auc"]:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Guess (AUC=0.50)')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Comparison — All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.savefig('roc_all_models.png', dpi=150, bbox_inches='tight')
plt.show()

---

# 🎯 STEP 8: Feature Importance

## What Is Feature Importance?

Feature importance answers: **"Which inputs did the model rely on most to make predictions?"**

This is powerful for business because:
1. You know which data to invest in collecting better
2. You can tell teams what to act on ("fix outages" vs. "lower bills")
3. It validates that the model learned sensible things (not spurious patterns)
4. You can remove low-importance features to simplify the model

In [ ]:
# Random Forest feature importance (based on how much each feature reduces impurity)
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(9, 7))
colors_imp = ['tomato' if v > importances.quantile(0.7) else 'steelblue' for v in importances.values]
bars = plt.barh(importances.index, importances.values, color=colors_imp, edgecolor='black', alpha=0.85)
plt.xlabel('Feature Importance (Gini)', fontsize=12)
plt.title('Random Forest — Feature Importance', fontsize=14, fontweight='bold')
plt.axvline(x=0.05, color='red', linestyle='--', alpha=0.5, label='5% threshold')
plt.legend()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 most important features:')
top5 = importances.sort_values(ascending=False).head(5)
for feat, imp in top5.items():
    print(f'  {feat:<30} {imp:.4f} ({imp*100:.1f}% of total importance)')

### 📝 Business Translation of Feature Importance

The model has told us what drives churn. Now translate that to action:

| Top Feature | Business Meaning | Action |
|---|---|---|
| `tenure_months` | New customers churn most | Invest in onboarding programs for first 90 days |
| `monthly_charges` | High-cost customers at risk | Proactive discount offers for high-bill customers |
| `num_support_calls` | Frustrated callers leave | Escalate customers with 3+ calls to VIP support |
| `contract_type_Month-to-Month` | No commitment = easy exit | Incentivize annual contract upgrades |
| `num_service_outages` | Quality drives loyalty | SLA improvements for outage-prone regions |

**This is the difference between a data project and a business project.** The model is only useful if someone acts on it.

---

# 💡 STEP 9: Making Predictions on New Customers

In [ ]:
# Predict churn risk for 3 hypothetical new customers
# In production: this data comes from your CRM in real-time

new_customers = pd.DataFrame({
    'tenure_months':      [3,   48,  12],
    'monthly_charges':    [95,  45,  70],
    'total_charges':      [285, 2160, 840],
    'num_support_calls':  [5,   0,   2],
    'num_service_outages':[3,   0,   1],
    'age':                [25,  55,  35],
    'has_streaming':      [1,   0,   1],
    'has_security_addon': [0,   1,   0],
    # contract types
    'contract_type_Month-to-Month': [1, 0, 1],
    'contract_type_One-Year':       [0, 1, 0],
    'contract_type_Two-Year':       [0, 0, 0],
    # payment methods
    'payment_method_Bank Transfer':    [0, 1, 0],
    'payment_method_Credit Card':      [0, 0, 0],
    'payment_method_Electronic Check': [1, 0, 0],
    'payment_method_Mailed Check':     [0, 0, 1],
})

# Reorder columns to match training data
new_customers = new_customers[X_train.columns]

# Predict using Random Forest (best model)
churn_predictions = rf_model.predict(new_customers)
churn_probabilities = rf_model.predict_proba(new_customers)[:, 1]

profiles = [
    'New customer (3 mo), $95/mo, 5 support calls, Month-to-Month',
    'Loyal customer (48 mo), $45/mo, 0 calls, One-Year contract',
    'Mid-tenure (12 mo), $70/mo, 2 calls, Month-to-Month',
]

print('=== CHURN RISK PREDICTIONS FOR NEW CUSTOMERS ===')
print()
for i, (profile, pred, prob) in enumerate(zip(profiles, churn_predictions, churn_probabilities)):
    risk_level = '🔴 HIGH RISK' if prob > 0.6 else '🟡 MEDIUM RISK' if prob > 0.35 else '🟢 LOW RISK'
    action = 'URGENT: Assign retention team now' if prob > 0.6 else \
             'MONITOR: Send proactive offer'     if prob > 0.35 else \
             'OK: No immediate action needed'
    print(f'Customer {i+1}: {profile}')
    print(f'  Churn Probability: {prob:.1%}  →  {risk_level}')
    print(f'  Recommended Action: {action}')
    print()

---

# 🏢 ENTERPRISE CONTEXT: How Is This Used in Production?

## The Real ML Pipeline at a Company Like Jio or Airtel

```
DATA SOURCES
┌─────────────────────────────────────────────────────────────┐
│  CRM Database  │  Billing System  │  Support Tickets  │ App │
└─────────────────────────────────────────────────────────────┘
                          │
                          ▼
              DATA WAREHOUSE (Snowflake / BigQuery)
                          │
                          ▼
              FEATURE PIPELINE (daily batch job)
              - Compute tenure, call counts, charges
              - Handle missing values
                          │
                          ▼
              ML TRAINING PIPELINE (weekly retrain)
              - Retrain Random Forest on last 6 months
              - Compare against previous model
              - Auto-promote if better
                          │
                          ▼
              MODEL SERVING (REST API)
              POST /predict {customer_id: 'CUST_00123'}
              Response: {churn_prob: 0.73, risk: 'HIGH'}
                          │
                    ┌─────┴──────┐
                    ▼            ▼
             CRM DASHBOARD    AUTOMATED EMAIL
             (sales team)     (retention offer)
```

## Key Questions Asked in Enterprise ML Interviews

| Question | What They're Really Asking |
|---|---|
| "How do you handle imbalanced data?" | Do you know SMOTE, class weights, threshold tuning? |
| "How do you prevent overfitting?" | Cross-validation, regularization, early stopping |
| "How do you evaluate a churn model?" | Do you know recall vs precision tradeoff? |
| "What if your model degrades over time?" | Data drift monitoring, periodic retraining |
| "How do you explain the model to a business stakeholder?" | Feature importance, SHAP values, simple visualizations |

---

# 📝 BONUS: Cross-Validation — A Better Way to Evaluate

## The Problem With a Single Train/Test Split

Your 80/20 split might be lucky or unlucky. If the test set happened to contain easier examples, your accuracy looks better than it is.

**Cross-Validation (CV)** solves this by training and testing on ALL the data in rotation.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# 5-Fold Cross Validation:
# Split data into 5 equal parts
# Train on 4 parts, test on 1 → repeat 5 times
# Result: 5 accuracy scores → take the average

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Running 5-Fold Cross-Validation (this gives a more reliable estimate)...\n')
print(f'{"Model":<22} {"CV Mean AUC":>12} {"CV Std":>10} {"Min":>8} {"Max":>8}')
print('-' * 65)

models_to_cv = [
    ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42), True),
    ('Decision Tree',       DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=42), False),
    ('Random Forest',       RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42), False),
]

for name, model, needs_scale in models_to_cv:
    X_cv = X_train_scaled if needs_scale else X_train
    scores = cross_val_score(model, X_cv, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f'{name:<22} {scores.mean():>12.4f} {scores.std():>10.4f} {scores.min():>8.4f} {scores.max():>8.4f}')

print('\n💡 Lower std = more stable model = less dependent on random luck in the split')

---

# 🏁 SUMMARY — What You Learned in Module 1

## Concepts Covered

### Machine Learning Fundamentals
- ✅ **Supervised learning**: train on labeled data, predict on new data
- ✅ **Features (X)** vs **Target (y)**: inputs vs what we predict
- ✅ **Training set** vs **Test set**: why we never touch test data during training
- ✅ **Overfitting**: memorizing training data instead of learning real patterns

### The 7-Step ML Workflow
1. ✅ Define the problem (predict customer churn)
2. ✅ Collect/generate data (5000 customers)
3. ✅ Explore data (EDA, distributions, correlations)
4. ✅ Preprocess (encode categories, scale, split)
5. ✅ Train models (3 algorithms)
6. ✅ Evaluate models (accuracy, AUC, precision, recall)
7. ✅ Use predictions for business action

### Algorithms
| Algorithm | Best For | Key Hyperparameter |
|---|---|---|
| Logistic Regression | Linear relationships, explainability | `C` (regularization strength) |
| Decision Tree | Explainable rules, non-linear | `max_depth` |
| Random Forest | Best accuracy on tabular data | `n_estimators`, `max_depth` |

### Evaluation Metrics
| Metric | When to Use |
|---|---|
| **Accuracy** | Balanced classes, quick sanity check |
| **Precision** | False alarms are costly (spam filter) |
| **Recall** | Missing positives is costly (disease, churn) |
| **F1 Score** | When both precision and recall matter |
| **AUC-ROC** | Best overall metric for classification |

---

## 🎓 Key Interview Answers You Can Now Give

**Q: "Walk me through an ML project you've done."**  
A: "I built a customer churn prediction model for a telecom company. I started with EDA to understand patterns, then preprocessed the data — one-hot encoding categoricals, scaling numerics. I trained three models — logistic regression as a baseline, then decision tree and random forest. Random Forest achieved AUC of X and recall of Y% on the held-out test set. I used feature importance to give the business team actionable insights: new customers with month-to-month contracts and high support call volumes were highest risk."

**Q: "What's the difference between precision and recall?"**  
A: "Precision is 'of the ones I flagged as churn, how many actually did?' Recall is 'of all the ones who actually churned, how many did I catch?' For churn, I'd optimize for recall — missing a churner costs more than wasting a retention offer."

**Q: "How do you handle overfitting in a Decision Tree?"**  
A: "Limit the tree depth with max_depth, set a minimum number of samples per leaf with min_samples_leaf, and validate using cross-validation rather than a single train/test split."

---

## 🚀 What's Next — Module 2 Preview

**Module 2: Deep Learning Fundamentals with PyTorch**

You'll learn:
- How a neural network actually learns (forward pass, backpropagation)
- What tensors, layers, and activation functions are
- Build a neural network from scratch that beats Random Forest on this same churn dataset
- Why deep learning is needed (and when it's NOT needed)

The bridge: Random Forests have fixed capacity. Neural networks can learn arbitrarily complex patterns — which is what LLMs like GPT and Claude are built on.

---

## 📚 Further Reading (All Free)

- **scikit-learn documentation**: https://scikit-learn.org/stable/user_guide.html
- **Hands-On Machine Learning (Aurélien Géron)**: Best book for traditional ML
- **fast.ai Practical ML**: Free course, very enterprise-focused
- **Kaggle Learn**: Free micro-courses with real datasets

---

*Module 1 of 6 — Fine-Tuning & ML Fundamentals Series*  
*Next: Module 2 — Deep Learning Fundamentals (PyTorch)*